# 全天パイプライン: 1年スキャンから (I,Q,U) マップまで

このnotebookは、EBCoDeCMB.jl のパイプライン全体を一度に通します。
nside 16、lmax 47、Falcons のデフォルトスキャンを 1 Hz で 1 年実行し、
リポジトリに同梱した小さな調和係数を入力として使います。旧実装との比較は、
検証結果ファイルを別途持っている場合だけ最後の節で実行できます。

```
ポインティング (φ, θ, ψ) のチャンク
    |  build_ring_scan_buffer          スキャン全体で 1 回
RingScanBuffer  (ピクセル順)
    |  convolve_sphere_gpu / _multigpu  リングごと
accumulated[sky, beam, active_pixel, scan_weight]
    |  binning_mapmake!
stokes_map[sky, beam, (I,Q,U), pixel]
```

必要なもの: CUDA が使えるGPU 1台以上。実行時間は全体で数分です。

複数GPUを本当に並列に回すには、Julia のスレッド数をデバイス数以上にしてください。
Jupyter を起動する前に `export JULIA_NUM_THREADS=8` などとしておきます。
足りない場合はワーカーが同じスレッド上で交互に走り、警告が出ます (結果は変わりません)。


## 0. セットアップ

`notebook/Project.toml` に Falcons や NPZ などnotebook専用の依存が入っています。
パッケージ本体の `Project.toml` はこれらに依存しません。


In [ ]:
using Pkg

function find_repo_root(start=pwd())
    current = abspath(start)
    while true
        if isfile(joinpath(current, "Project.toml")) &&
            isfile(joinpath(current, "src", "EBCoDeCMB.jl"))
            return current
        end
        parent = dirname(current)
        parent == current && error("EBCoDeCMB.jl repository root was not found")
        current = parent
    end
end

const REPO = find_repo_root()
Pkg.activate(joinpath(REPO, "notebook"))

using EBCoDeCMB, CUDA, Falcons, Healpix, NPZ, JLD2
using Printf, Statistics, LinearAlgebra

@printf("CUDA devices: %d,  Julia threads: %d\n",
        length(CUDA.devices()), Threads.nthreads())


## 1. ConvolutionParams

畳み込み全体の設定です。ここで決まる主な量:

- `lstart:lstop` が畳み込む多重極、`lmax` は入力 alm/blm の帯域
- `bnmax` はビームの方位次数 `n` の上限
- `κ` は `|M − n| ≤ κ` の打ち切り。`Mmax = min(κ + bnmax, lmax)` が自動で決まります
- `numofsky` / `numofbeam` は同時に処理する空とビームの本数

今回は 1 空 × 1 ガウスビームです。


In [ ]:
nside = 16
lmax  = 3nside - 1      # 47

cp = ConvolutionParams(
    nside_output = nside,
    lstart       = 0,
    lstop        = lmax,
    lmax         = lmax,
    bnmax        = 5,
    κ            = 10,
    numofsky     = 1,
    numofbeam    = 1,
)
show_cp(cp)
@printf("\nMmax = %d,  リング数 = %d,  ピクセル数 = %d\n",
        calc_Mmax(cp, cp.lstop), ring_count(cp), cp.resol.numOfPixels)


## 2. スキャンを RingScanBuffer に流し込む

`build_ring_scan_buffer` は **`(φ, θ, ψ)` のチャンクを列挙できるものなら何でも** 受け取ります。
スキャン戦略にも生成方法にも依存しません。ここでは Falcons を使いますが、
ファイルから読んでも、別のシミュレータでも同じです。

チャンクで渡すのが要点で、3153万サンプル分のポインティングを一度にメモリに置かずに済みます。
バッファは全サンプルを **ピクセル順** に並べ替えるので、以降どのリングも連続領域を切り出すだけになります。

> リングごとに時系列全体を走査する `make_ring_angles(cp, geometry, φ, θ, ψ)` もありますが、
> それは 1 リング用です。全天だと `O(サンプル数 × リング数)` になります。


In [ ]:
ss = gen_ScanningStrategy()   # Falcons のデフォルト (α=45°, β=50°, spin 0.05 rpm)
ss.nside         = nside
ss.sampling_rate = 1.0
ss.duration      = 365 * 24 * 60 * 60
nptg  = round(Int, ss.duration * ss.sampling_rate)
chunk = 30 * 24 * 60 * 60      # 30日ずつ生成する

pointing_chunks = Channel{NTuple{3,Vector{Float64}}}(2)
producer = @async begin
    for start in 0:chunk:(nptg - 1)
        stop = min(nptg, start + chunk)
        theta, phi, psi, _ = get_pointings(ss, start, stop)
        put!(pointing_chunks, (vec(phi[:, 1]), vec(theta[:, 1]), vec(psi[:, 1])))
    end
    close(pointing_chunks)
end

elapsed = @elapsed buffer = build_ring_scan_buffer(
    cp, pointing_chunks; capacity_hint = nptg,
)
wait(producer)

@printf("%.1f s   %d サンプル保持   %.2f GiB   (%.1f bytes/sample)\n",
        elapsed, length(buffer), ring_scan_buffer_bytes(buffer)/2^30,
        ring_scan_buffer_bytes(buffer)/length(buffer))
buffer


リングを取り出すと `(ActiveRingPixels, Vector{Angles})` が返ります。
これは 1 リング版の `make_ring_angles` とまったく同じ契約で、そのまま `convolve_ring_gpu` に渡せます。


In [ ]:
geometry = ring_geometry(cp, 32)
active, angles = make_ring_angles(cp, geometry, buffer)
@printf("リング %d: %d/%d ピクセルがヒット, 合計 %d サンプル, complete=%s\n",
        geometry.ring, length(active), geometry.pixel_count,
        sum(a.num for a in angles), active.complete)
EBCoDeCMB._free_active_ring_pixels!(active)


## 3. 空とビームを用意する

入力は healpy と同じ詰め方の `(3, (lmax+1)(lmax+2)/2)` 行列で、行は `(T, E, B)` です。
`prepare_alm` / `prepare_blm` がスピン基底 `(0, +2, −2)` に変換し、負の `m` を再構成します。

この例では、リポジトリの `test/data` に同梱した CMB sky とガウスビームを使います。


In [ ]:
inputs = joinpath(REPO, "test", "data")
alm_teb = npzread(joinpath(inputs, "alm_cmb_r0_nside16.npy"))
blm_gauss = npzread(joinpath(inputs, "blm_gauss_fwhm5_nside16.npy"))

prepared_alm = prepare_alm(cp, alm_teb; precision = Float64)
prepared_blm = prepare_blm(cp, blm_gauss; precision = Float64)

println(prepared_alm)
println(prepared_blm)


## 4. β の範囲と Wigner–Chebyshev テーブル

`β` は「実際に向いた方向」と「ピクセル中心」の角距離です。`ScanMomentPlan` の
`beta_max` は **スキャン中のすべての β を含まないといけません**。超えると `DomainError` になります。

テーブルと scan moments は必ず同じ plan から作ってください。異なる `beta_max` から作ると
Chebyshev 変数が共有されません。`convolve_ring_gpu` はこれを検査して弾きます。


In [ ]:
maxbeta = 0.0
for ring in 1:ring_count(cp)
    a, angs = make_ring_angles(cp, ring_geometry(cp, ring), buffer)
    a === nothing && continue
    global maxbeta = max(maxbeta, maximum(x -> maximum(x.β), angs))
    EBCoDeCMB._free_active_ring_pixels!(a)
end
@printf("max β = %.4e rad   (ピクセルスケール %.4e rad)\n",
        maxbeta, sqrt(4pi / cp.resol.numOfPixels))

scan_plan = ScanMomentPlan(
    kappa      = cp.κ,
    nmax       = calc_nmax(cp, cp.lstop),
    beta_order = 16,
    beta_max   = 1.02 * maxbeta,
    precision  = Float64,
)
table = @time build_wigner_chebyshev_table_cpu(cp, scan_plan)
@printf("テーブル: %d 個の (M,n) ペア × %d 個の l × Chebyshev %d 次\n",
        length(table.pair_M), table.lstop - table.lstart + 1,
        size(table.coefficients, 1))


## 5. メモリ見積もりと l タイル幅

`ring_memory_bytes` はリング 1 本が確保するデバイスメモリの内訳を返します。
テストで実際の確保サイズとバイト単位で照合してあります。

`suggest_l_tile` は Int32 インデックス上限と空きメモリの両方から最大のタイル幅を返します。
`convolve_sphere_gpu` の `l_tile` は既定で `:auto` なので通常は明示不要ですが、
大きな問題で事前に見積もるときに使えます。


In [ ]:
ring_pixels = 4nside                      # 赤道リングのピクセル数
est = ring_memory_bytes(cp, ring_pixels, ring_pixels, 1; complete_ring = true)
@printf("リング常駐: ringd %.2f MiB, 合計 %.2f MiB\n",
        est.ringd/2^20, est.persistent/2^20)
@printf("l タイル 1 本あたり: %.2f MiB\n", est.tile_peak/2^20)
@printf("Int32 上限のタイル幅: %d\n", max_indexable_l_tile(cp, ring_pixels))
@printf("suggest_l_tile      : %d\n",
        suggest_l_tile(cp, ring_pixels, ring_pixels; complete_ring = true))


## 6. 全天を畳み込む

`convolve_sphere_multigpu` はリングを共有キューから 1 本ずつ配ります。極と赤道でリングの
コストが 1 桁以上違うため、事前分割ではなくキューにしています。

ワーカーはデバイスごとに自分の `GPUAlm`・Wigner テーブル・`RingDContext` を持つので、
引数はホスト側の `PreparedAlm` と `WignerChebyshevTable` を渡します。
1 台だけで走らせるなら `convolve_sphere_gpu` にアップロード済みのものを渡す形になります。


In [ ]:
devices = collect(0:(length(CUDA.devices()) - 1))
progress = NamedTuple[]

elapsed = @elapsed maps = convolve_sphere_multigpu(
    cp, buffer, prepared_alm, prepared_blm, table, scan_plan;
    devices           = devices,
    map_precision     = Float64,
    reclaim_interval  = 16,
    progress_callback = record -> push!(progress, record),
)

@printf("%.1f s で %d リング (devices=%s)\n", elapsed,
        progress[end].processed_rings, devices)
@printf("maps: %s   有限ピクセル %d/%d\n", size(maps),
        count(isfinite, maps[1,1,1,:]), cp.resol.numOfPixels)


`maps[sky, beam, stokes, pixel]` です。ここでは 1 sky × 1 beam の結果を確認します。


In [ ]:
for (i, component) in enumerate(("I", "Q", "U"))
    values = maps[1, 1, i, :]
    finite_values = values[isfinite.(values)]
    @printf("%s rms = %.6g\n", component, std(finite_values))
end


軸対称ビームなので、`I` は「調和空間でビーム窓を掛けた空」と一致するはずです。
Healpix.jl で独立に合成して比べます (畳み込み定理の `√(4π/(2l+1))` が入ります)。


In [ ]:
nalm = numberOfAlms(lmax, lmax)
T = Alm(lmax, lmax, zeros(ComplexF64, nalm))
packed(l, m) = m*(lmax+1) - (m*(m-1))÷2 + (l-m) + 1
for l in 0:lmax, m in 0:l
    T.alm[almIndex(T, l, m)] =
        alm_teb[1, packed(l, m)] * sqrt(4pi/(2l+1)) * blm_gauss[1, packed(l, 0)]
end
smoothed = alm2map(T, nside)

I = maps[1, 1, 1, :]
@printf("畳み込み I rms = %.4f\n", std(I))
@printf("平滑化空 I rms = %.4f   (差は β≠0 のポインティングオフセット由来)\n",
        std(smoothed.pixels))


## 7. 出力モードを使い分ける

全天マップは常に確保できるとは限りません。nside 512 で 22 空 × 100 ビームだと
Float32 でも約 83 GB になります。

- `:full_map` — 全天マップを返す (既定)
- `:ring_callback` — リングごとに `(sky, beam, stokes, active_pixel)` を渡して解放
- `:discard` — 計算して捨てる。GPU 経路だけのベンチマーク用


In [ ]:
ring_rms = Dict{Int,Float64}()
convolve_sphere_multigpu(
    cp, buffer, prepared_alm, prepared_blm, table, scan_plan;
    devices       = devices,
    output        = :ring_callback,
    ring_callback = payload -> (ring_rms[payload.ring] =
        std(payload.stokes[1, 1, 1, :])),
)
@printf("%d リング分を受け取りました。最初の5本の I rms:\n", length(ring_rms))
for ring in sort(collect(keys(ring_rms)))[1:5]
    @printf("  ring %-3d  %.4f\n", ring, ring_rms[ring])
end


## 8. チェックポイントと再開

`:full_map` では途中経過をファイルに残せます。書き込みは一時ファイル経由の
rename なので、中断で壊れたファイルが残ることはありません。
再開時は設定と `checkpoint_id` の署名を照合し、合わない場合は例外にします。
`checkpoint_id` は alm・blm・pointing の内容を変えるたびに更新してください。


In [ ]:
checkpoint = joinpath(mktempdir(), "tutorial.ckpt")
checkpoint_id = "full-sphere-tutorial-v1"

# 途中で中断されたことにする
try
    convolve_sphere_multigpu(
        cp, buffer, prepared_alm, prepared_blm, table, scan_plan;
        devices             = devices,
        checkpoint_path     = checkpoint,
        checkpoint_id       = checkpoint_id,
        checkpoint_interval = 8,
        progress_callback   = r -> r.ring_index == 40 && error("中断"),
    )
catch err
    println("中断: ", err)
end

partial = load_sphere_checkpoint(checkpoint, cp)
@printf("チェックポイント: %d/%d リング完了\n",
        count(partial.completed), length(partial.completed))

resumed = convolve_sphere_multigpu(
    cp, buffer, prepared_alm, prepared_blm, table, scan_plan;
    devices         = devices,
    checkpoint_path = checkpoint,
    checkpoint_id   = checkpoint_id,
)
finite = isfinite.(maps)
@printf("再開後のマップは通しの結果と一致: %s\n",
        resumed[finite] == maps[finite])


## 9. 旧実装との照合

旧実装の検証結果を持っている場合は、環境変数 `EBCODE_LEGACY_RESULT` にその
JLD2ファイルを指定すると比較できます。指定しなければ、この節は安全にスキップします。


In [ ]:
legacy_path = get(ENV, "EBCODE_LEGACY_RESULT", "")
if isempty(legacy_path)
    @info "EBCODE_LEGACY_RESULT is not set; skipping the legacy comparison"
else
    legacy = JLD2.load(legacy_path)["maps_standard"]
    for (i, component) in enumerate(("I", "Q", "U"))
        ref = legacy[1, 1, i, :]
        got = maps[1, 1, i, :]
        finite = isfinite.(ref) .& isfinite.(got)
        delta = got[finite] .- ref[finite]
        @printf("%s: max|diff|=%.4e, rel L2=%.4e\n", component,
                maximum(abs, delta), norm(delta) / norm(ref[finite]))
    end
end


## 10. 注意点

- **`rcond`**: `binning_mapmake!` は `1/cond(H) ≤ rcond` のピクセルを `unseen` にします。
  既定の `1e-10` は旧実装から引き継いだ値で、`cond ≈ 1e10` のピクセルを通してしまいます。
  ψ 被覆が悪いピクセルがあるスキャンでは、既定値のままだと少数の外れ値がマップ統計を壊します。
  実データでは条件数の分布を確認して決めてください。
- **`beta_max`**: スキャン全体の最大 β を含める必要があります。テーブルと scan moments は
  必ず同じ plan から作ってください。
- **HWP**: 現在は `HWP=false` のみ対応です。スキャンバッファも `(φ, θ, ψ)` しか保持しません。
- **精度**: `Float32` と `Float64` の両方に対応しています。本番規模では `Float32` が既定の選択です。
